In [3]:
from ase.io import read, write
import os

cif_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/Python Packages test/MOFs_to_pre_process'
cifs = os.listdir(cif_path)
cifs.sort()

for cif in cifs:
	
    mof = read(os.path.join(cif_path, cif))
	
    write(os.path.join(cif_path, cif), mof)



UnknownFileTypeError: 

In [4]:
from ase.io import read
import os
import numpy as np

cutoff = 0.75  # interatomic distance threshold
folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/Python Packages test/MOFs_to_pre_process'
bad_list = []
for cif in os.listdir(folder):
    mof = read(os.path.join(folder, cif))
    d = mof.get_all_distances()
    upper_diag = d[np.triu_indices_from(d, k=1)]
    for entry in upper_diag:

        if entry < cutoff:

            print('Interatomic distance issue:' + cif.split('.')[0])

            bad_list.append(cif)

            break

with open('bad_cifs_distance_check.txt','w') as w:

    for bad_cif in bad_list:

        w.write(bad_cif+'\n')

Interatomic distance issue:MOF808_O
Interatomic distance issue:MOF 806


In [3]:
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env
from pymatgen.core import Structure
import os

folder = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/MOFs_to_pre_process'
cifs = os.listdir(folder)
cifs.sort()
bad_list = []
for cif in cifs:
	
    mof = Structure.from_file(os.path.join(folder, cif))
	
    nn = local_env.CrystalNN()
	
    graph = StructureGraph.with_local_env_strategy(mof, nn)
	
    for j in range(len(mof)):
		
        nbr = graph.get_connected_sites(j)
		
        if not nbr:
			
            print('Lone atom issue:' + cif+'\n')
			
            bad_list.append(cif)
			
            break

with open('bad_cifs_lone_atom_check.txt','w') as w:
	
    for bad_cif in bad_list:
		
        w.write(bad_cif+'\n')

AttributeError: specie

In [ ]:
from pymatgen.core import Structure
from pymatgen.analysis import structure_matcher
import os

folder = r'' #folder of CIFs to de-duplicate
new_folder = r'/new/folder/to/save/cifs' #folder to save only unique CIFs

mofs = [] #initialize list to store Pymatgen structures
entries = os.listdir(folder) #get all CIFs
entries.sort() #alphabetical sort

#for every CIF, store Pymatgen Structure in list
for entry in entries:

    if '.cif' not in entry:
        continue
    
    #read CIF
    mof_temp = Structure.from_file(os.path.join(folder,entry),primitive=False)

    #tag Pymatgen structure with its name
    mof_temp.name = entry
    mofs.append(mof_temp)

#Initialize StructureMatcher
sm = structure_matcher.StructureMatcher(primitive_cell=True)

#Group structures
groups = sm.group_structures(mofs)
print(str(len(groups))+' unique out of '+str(len(entries))+' total')

#Write out set of only unique CIFs
if not os.path.exists(new_folder):
    os.mkdir(new_folder)
for group in groups:
    mof_temp = group[0]
    mof_temp.to(filename=os.path.join(new_folder,mof_temp.name))

In [4]:
from ase import neighborlist
from ase.io import read
import os
import numpy as np
import warnings

# Metals that should not have terminal oxo ligands
metals = ['Li','Na','K','Rb','Cs','Fr',

          'Be','Mg','Ca','Sr','Ba','Ra',

          'Sc','Y','La','Ac',

          'Ti','Zr','Hf',

          'Mn',

          'Fe',

          'Co',

          'Ni',

          'Cu','Ag',

          'Zn','Cd',

          'Al','Ga','In','Tl']

# Path to CIFs
p = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/MOFs_to_pre_process'

# Get CIFs from folder
cifs = os.listdir(p)
cifs = [cif for cif in cifs if '.cif' in cif]
cifs.sort()

# Check every CIF
bad_list = []
for cif in cifs:


    bad = False


    # Read in CIF, ignoring ASE warnings

    with warnings.catch_warnings():

        warnings.simplefilter('ignore')

        structure = read(os.path.join(p,cif))


    # Get list of atomic symbols

    syms = np.array(structure.get_chemical_symbols())


    # Is one of the specified metals in this MOF

    if not any(item in syms for item in metals):

        continue


    # Initialize neighbor list

    cutoff = neighborlist.natural_cutoffs(structure)

    nl = neighborlist.NeighborList(cutoff,self_interaction=False,bothways=True)

    nl.update(structure)

	
    # For every site, check if it is a terminal metal-oxo
	
    for i, sym in enumerate(syms):

		
        # Confirm site is in pre-specified metal list
		
        if sym not in metals:
			
            continue

		
        # Get neighbors to metal
		
        bonded_atom_indices = nl.get_neighbors(i)[0]
		
        if bonded_atom_indices is None:
			
            continue
		
        bonded_atom_symbols = syms[bonded_atom_indices]

		
        # For every neighbor, check if it's a terminal oxo
		
        for j, bonded_atom_symbol in enumerate(bonded_atom_symbols):

			
            # Confirm neighbor is an O atom
			
            if bonded_atom_symbol != 'O':
				
                continue

			
            # Check if the O atom is only bound to the metal
			
            cn = len(nl.get_neighbors(bonded_atom_indices[j])[0])
			
            if cn == 1:
				
                bad = True
				
                print('Missing H on terminal oxo: ' + cif)
				
                bad_list.append(cif)

			
            if bad:
				
                break
		
        if bad:
			
            break

with open('bad_cifs_oxo_check.txt','w') as w:
	
    for bad_cif in bad_list:
		
        w.write(bad_cif + '\n')

In [5]:
from ase.io import read, write
import os

cif_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/MOFs_to_pre_process'
cifs = os.listdir(cif_path)
cifs.sort()

refcodes = []
mofs = []
for cif in cifs:
	
    refcodes.append(cif.split('.cif')[0])
	
    mofs.append(read(os.path.join(cif_path, cif)))
write('mofs.xyz', mofs)

with open('refcodes.csv','w') as w:
	
    for refcode in refcodes:
		
        if refcode == refcodes[-1]:
			
            w.write(refcode)
		
        else:
			
            w.write(refcode+',')

In [2]:
from ase.io import read, write
import numpy as np
import os

# Converts an appended .xyz to a folder of CIFs

# Relevant filenames
refcode_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/refcodes.csv' # path to refcodes
xyz_path = r'/Users/shubhamjamdade/Desktop/MOF_ready_for_water_simulations/mofs.xyz' # path to XYZ of all structures
new_folder = r'cifs' # path to new folder store CIFs

# ----------------------
refs = np.genfromtxt(refcode_path,delimiter=',',dtype=str)
mofs = read(xyz_path,index=':')

if not os.path.exists(new_folder):
	
    os.mkdir(new_folder)
for i, mof in enumerate(mofs):
	
    write(os.path.join(new_folder,refs[i]+'.cif'),mof)